## LFR graph generation

In [1]:
import networkx as nx

n = 1000
tau1 = 2  # Power-law exponent for the degree distribution
tau2 = 1.1  # Power-law exponent for the community size distribution
avg_degree = 25  # Average Degree
max_degree = int(0.1 * n)  # Max Degree
min_community = 60  # Min Community Size
max_community = int(0.1 * n)  # Max Community Size
MU = 0.1
# MU = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5]

G = nx.generators.community.LFR_benchmark_graph(
    n, tau1, tau2, MU, average_degree=avg_degree, max_degree=max_degree, min_community=min_community, max_community=max_community,
    seed=17
)

# Remove multi-edges and self-loops from G
G = nx.Graph(G)
selfloop_edges = list(nx.selfloop_edges(G))
G.remove_edges_from(selfloop_edges)

In [2]:
import numpy as np
# Map each unique set to community ID
community_sets = list({frozenset(G.nodes[n]['community']) for n in G.nodes})
community_id_map = {com: idx for idx, com in enumerate(community_sets)}

# Create a node → community ID mapping
com_labels = {n: community_id_map[frozenset(G.nodes[n]['community'])] for n in G.nodes}
true_labels = np.array([com_labels[i] for i in range(n)])

In [3]:
# Visualize the graph
# import matplotlib.pyplot as plt

# unique_coms = list(set(com_labels.values()))
# color_map = {com: color for com, color in zip(unique_coms, plt.cm.tab20.colors)}
# node_colors = [color_map[com_labels[n]] for n in G.nodes]

# plt.figure(figsize=(10, 10))
# pos = nx.spring_layout(G, seed=42)
# nx.draw_networkx_nodes(G, pos, node_size=10, node_color=node_colors, alpha=0.6)
# nx.draw_networkx_edges(G, pos, width=0.1, alpha=0.3)
# plt.title(f"LFR {n} nodes with {len(unique_coms)} Communities ")
# plt.axis('off')
# plt.show()

for quick test let's create features using random generator

In [4]:
import numpy as np

# Randomly generate node features
num_features = 32
features = np.random.randn(len(G.nodes), num_features)


In [5]:
import torch
from torch_geometric.utils import from_networkx

# Add features to NetworkX nodes
for i, feat in enumerate(features):
    G.nodes[i]['x'] = torch.tensor(feat, dtype=torch.float)

# Convert to PyG format
data = from_networkx(G)
data.x = torch.stack([data.x[i] for i in range(data.num_nodes)])  # ensure proper shape


In [6]:
data

Data(x=[1000, 32], edge_index=[2, 36228], community=[1000])

In [15]:
import dmon
num_clusters = len(set(com_labels.values()))
model = dmon.DMoN(in_channels=num_features, num_clusters=num_clusters)  # 1-layer DMoN pooling

learning_rate = 0.01
weight_decay = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [16]:
# Train the model
for epoch in range(201):
    model.train()
    optimizer.zero_grad()
    ca, loss = model(data.x, data.edge_index)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")

Epoch 000 | Loss: 1.1783
Epoch 020 | Loss: 0.1313
Epoch 040 | Loss: 0.1279
Epoch 060 | Loss: 0.1645
Epoch 080 | Loss: 0.1009
Epoch 100 | Loss: 0.1486
Epoch 120 | Loss: 0.1139
Epoch 140 | Loss: 0.1419
Epoch 160 | Loss: 0.1064
Epoch 180 | Loss: 0.1004
Epoch 200 | Loss: 0.1473


In [9]:
model.eval()
ca, _ = model(data.x, data.edge_index)
pred_labels = ca.argmax(dim=1).numpy()

print(f"Shape of predicted labels: {pred_labels.shape}")

Shape of predicted labels: (1000,)


In [10]:
from sklearn.metrics import normalized_mutual_info_score

nmi = normalized_mutual_info_score(true_labels, pred_labels)

print(f"NMI between DMoN and LFR ground truth: {nmi:.4f}")


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


NMI between DMoN and LFR ground truth: 0.0601
